# Missing Data — Edge Cases and Details

This notebook covers NaN edge cases and detailed propagation rules. For the core NaN convention and arithmetic behavior, see [Arithmetic Convention](arithmetic-convention.ipynb).

1. [NaN internals](#nan-internals) — where NaN lives, how it propagates
2. [NaN in arithmetic](#nan-in-arithmetic) — how NaN behaves in each operation
3. [Handling NaN with `.fillna()`](#handling-nan-with-fillna) — when you want different behavior
4. [Masking constraints](#masking-constraints) — `.sel()` and `mask=`
5. [Masking with NaN in coefficients](#masking-with-nan-in-coefficients) — multi-dimensional patterns
6. [Legacy NaN behavior](#legacy-nan-behavior-for-comparison) — how it worked before

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

import linopy

linopy.options["arithmetic_convention"] = "v1"

In [ ]:
m = linopy.Model()
time = pd.RangeIndex(5, name="time")
x = m.add_variables(lower=0, coords=[time], name="x")

# Data with NaN
data = xr.DataArray([1.0, np.nan, 3.0, 4.0, 5.0], dims=["time"], coords={"time": time})

---

## NaN internals

This section covers the internal mechanics of NaN handling. For the user-facing rules, see [Arithmetic Convention](arithmetic-convention.ipynb#nan-convention).

### How NaN enters

NaN can enter linopy data structures from three sources:

1. **`mask=` argument** at construction (`add_variables`, `add_constraints`) — you explicitly declare which slots exist.
2. **Structural operations** that produce absent slots: `.shift()`, `.where()`, `.reindex()`, `.reindex_like()`, `.unstack()` (with missing combinations).
3. **User-supplied data** with NaN values — NaN in multiplicative constants masks terms; NaN in additive constants is treated as 0.

Operations that do **not** produce NaN: `.roll()` (circular), `.sel()` / `.isel()` (subset), `.drop_sel()` (drops), `.expand_dims()` / `.broadcast_like()` (broadcast existing data).

### How NaN propagates

An expression is a sum of terms. Each term has a coefficient, a variable reference, and the expression has a shared constant. NaN marks an **individual term** as absent — it does not mask the entire coordinate.

When expressions are combined (e.g., `x*2 + y.shift(time=1)`), each term is kept independently. At time=0, `y.shift` contributes no term (NaN coeffs, vars=-1), but `x*2` is still valid. The result at time=0 is `2*x[0]` — not absent.

A coordinate is only fully absent when **all** terms have vars=-1 **and** the constant is NaN. This is exactly what `isnull()` checks.

### Where NaN lives

NaN is burned directly into the float fields: `coeffs`, `const`, `rhs`, `lower`, `upper`. Integer fields (`labels`, `vars`) use **-1** as their equivalent sentinel. There is no separate boolean mask array.

### Why this is consistent

- **`lhs >= rhs` is `lhs - rhs >= 0`**, so RHS obeys the same rule as any constant — no special case.
- **No dual role for NaN**: it always means "absent/nothing here." Internal NaN (from `shift`, `mask=`) and user NaN (from data) are treated identically.

- **Absent terms, not absent coordinates**: combining a valid expression with a partially-absent one does not destroy the valid part. Only when *every* term at a coordinate is absent is the coordinate itself absent.

---

## NaN in arithmetic

Under v1, NaN in arithmetic operands is handled automatically:

| Operation | NaN behavior | Rationale |
|---|---|---|
| `expr + nan_data` | NaN → 0 (additive identity) | Adding nothing = no contribution |
| `expr - nan_data` | NaN → 0 (additive identity) | Subtracting nothing = no change |
| `expr * nan_data` | NaN → absent (masks term) | Multiplying by nothing = no term |
| `expr / nan_data` | NaN → absent (masks term) | Dividing by nothing = no term |
| `expr <= nan_rhs` | **`ValueError`** | Silently skipping constraints is dangerous |

Constraint RHS is the one place where NaN still raises. This is intentional: `expr <= nan` could mean `expr <= 0` (treating NaN as 0 per the subtraction rule) or "no constraint" — both are plausible and both are dangerous if guessed wrong. Use `.sel()` or `mask=` to be explicit.

In [ ]:
# NaN in arithmetic — automatic handling
add_result = x + data
mul_result = x * data

print("add: NaN position const =", add_result.const.sel(time=1).item())  # 0.0
print("mul: NaN position absent?", mul_result.isnull().sel(time=1).item())  # True
print()

# Constraint RHS with NaN raises
try:
    x >= data
except ValueError as e:
    print("constraint: ValueError raised —", str(e)[:60])

---

## Handling NaN with `.fillna()`

NaN is handled automatically in most cases. Use `.fillna()` when the default behavior isn't what you want:

| Default behavior | If you want instead | Use |
|---|---|---|
| `expr * nan → absent` | Zero coefficient (term exists but contributes nothing) | `expr * data.fillna(0)` |
| `expr * nan → absent` | Keep original coefficient (no scaling) | `expr * data.fillna(1)` |
| `expr / nan → absent` | Keep original coefficient (no scaling) | `expr / data.fillna(1)` |
| `expr + nan → +0` | A specific fill value | `expr + data.fillna(value)` |

The key difference from legacy: `mul` and `div` now **mask** at NaN positions (making the term absent) rather than filling with 0 or 1. If you relied on the legacy fill behavior, add an explicit `.fillna()`.

In [ ]:
# Fill NaN before operating — you choose the fill value
print("add fillna(0):", (x + data.fillna(0)).const.values)
print("mul fillna(0):", (x * data.fillna(0)).coeffs.squeeze().values)
print("mul fillna(1):", (x * data.fillna(1)).coeffs.squeeze().values)

---

## Masking constraints

A common pattern: your data has NaN at positions where no constraint should exist. For example, availability data that's only defined for certain hours, or cost data with missing entries.

### Approach 1: `.sel()` (preferred)

Select only valid positions — the constraint has fewer coordinates:

In [ ]:
# Availability data with NaN = "no limit at this hour"
availability = xr.DataArray(
    [100.0, 80.0, np.nan, np.nan, 60.0], dims=["time"], coords={"time": time}
)

# Select only where data is valid — constraint has fewer coordinates
valid = availability.notnull()
m.add_constraints(x.sel(time=valid) <= availability.sel(time=valid), name="avail")

No fillna, no mask parameter — the constraint simply doesn't exist at the NaN positions.

### Approach 2: `mask=` parameter

When `.sel()` is inconvenient (e.g., multi-dimensional data where NaN positions vary across dimensions), use `mask=`:

In [ ]:
# Same result using mask= instead of .sel()
mask = availability.notnull()
m.add_constraints(x <= availability.fillna(0), name="avail_masked", mask=mask)

The same approaches work for variables with NaN bounds:

```python
# With .sel()
valid = upper_bounds.notnull()
m.add_variables(upper=upper_bounds.sel(i=valid), coords=[valid_coords], name="y")

# Or with mask=
mask = upper_bounds.notnull()
m.add_variables(upper=upper_bounds.fillna(0), mask=mask, name="y")
```

---

## Masking with NaN in coefficients

When NaN appears in coefficient data (e.g., efficiency factors where some combinations don't apply), the same two approaches work:

In [ ]:
# Efficiency data: solar has no efficiency at night (NaN)
techs = pd.Index(["solar", "wind"], name="tech")
hours = pd.RangeIndex(4, name="hour")
gen = m.add_variables(lower=0, coords=[hours, techs], name="gen")

efficiency = xr.DataArray(
    [[np.nan, 0.35], [0.8, 0.35], [0.9, 0.35], [np.nan, 0.35]],
    dims=["hour", "tech"],
    coords={"hour": hours, "tech": techs},
)

# Approach 1: .sel() — select only valid hours per tech
valid_hours = efficiency.sel(tech="solar").notnull()
solar_gen = gen.sel(tech="solar", hour=valid_hours)
solar_eff = efficiency.sel(tech="solar", hour=valid_hours)
print("sel approach — solar hours:", solar_gen.coords["hour"].values)

# Approach 2: mask= — keep all coordinates, mask invalid ones
rhs = xr.DataArray([50.0] * 4, dims=["hour"], coords={"hour": hours})
coeff_mask = efficiency.notnull().all("tech")
expr = gen * efficiency.fillna(0)
m.add_constraints(expr >= rhs, name="min_output", mask=coeff_mask)
print("mask approach — constraint mask:", coeff_mask.values)

---

## Legacy NaN behavior (for comparison)

Under legacy, NaN was handled implicitly:
- **In arithmetic**: silently replaced with neutral elements (0 for add/sub/mul, 1 for div)
- **In constraint RHS**: NaN meant "no constraint here" — auto-masked internally
- **With `auto_mask=True`**: NaN in variable bounds meant "no variable here"

This was convenient but could mask data quality issues. A NaN from a data pipeline bug would silently become 0, producing a valid but wrong model. The v1 convention makes NaN handling more transparent: NaN masks in mul/div (removing the term entirely) and contributes 0 in add/sub.

### Migration

| Legacy code | v1 behavior | Action needed? |
|---|---|---|
| `x + data_with_nans` | NaN → 0 (same effect) | None |
| `x * data_with_nans` | NaN → **absent** (legacy filled with 0) | If you wanted zero terms, use `.fillna(0)` |
| `x / data_with_nans` | NaN → **absent** (legacy filled with 1) | If you wanted identity, use `.fillna(1)` |
| `m.add_constraints(expr >= nan_rhs)` | NaN → constraint skipped (same effect) | None |
| `Model(auto_mask=True)` | Explicit `mask=` or `.sel()` | Same as before |

---

## Summary

| Aspect | v1 | Legacy |
|---|---|---|
| **NaN means** | Absent term (not absent coordinate) | Numeric placeholder (filled silently) |
| **NaN sources** | `mask=`, structural ops, user data | Same |
| **NaN in add/sub** | Treated as 0 (additive identity) | Same |
| **NaN in mul/div** | **Masks** the term (becomes absent) | Filled with 0 (mul) or 1 (div) |
| **NaN in constraint RHS** | `ValueError` (use `.sel()`/`mask=`) | Auto-masked (constraint skipped) |
| **Combining expressions** | Absent terms ignored, valid terms kept | NaN filled before combining |
| **Coordinate absent when** | All terms absent AND const is NaN | Never (NaN always filled) |
| **Masking** | Automatic via NaN in mul/div; explicit via `.sel()` or `mask=` | Implicit via NaN / `auto_mask` |
| **Storage** | Float fields + `-1` sentinels | Same, but NaN has dual role |
| **`.fillna()` needed?** | Only when you want non-default fill (e.g., `fillna(1)` for div) | Never (done automatically) |